# 마케팅 데이터 실전 — 결합 + 구조변환 (완성·정답본)

**매체노출**과 **온라인버즈** 두 데이터를 결합하고 구조변환해, 마케팅 성과를 분석합니다.
Day1에서 배운 걸 **실무형 데이터**로 한 번 더 적용해봐요.

> 이 실습의 핵심은 **그레인(한 행의 단위) 맞추기** 입니다 — 매체노출은 "하루 × 채널"(하루 4줄), 버즈는 "하루"(하루 1줄)라, 그냥 붙이면 안 되고 **집계로 단위를 맞춘 뒤** 결합합니다.

각 문제는 **문제 + 힌트 + `# 💡 풀이`** 형태입니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정 (OS 자동 감지)
import platform
_os = platform.system()
if _os == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif _os == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

## 🗂️ 데이터 소개 — 두 개의 마케팅 로그

- **매체노출**(`media`) : 캠페인을 **어디에(채널) 얼마나 노출**했나. **한 행 = 날짜 × 채널** → 하루에 채널 수(4)만큼 줄이 생김.
- **온라인버즈**(`buzz`) : 그날 온라인에서 **얼마나 언급되고 반응**했나(버즈량·감정·유입수). **한 행 = 날짜 1건**.

두 표를 붙이려면 **단위(그레인)를 맞춰야** 합니다.

In [ ]:
# 데이터 불러오기 (날짜는 날짜형으로)
media = pd.read_csv('data/매체노출_2024_2025.csv', parse_dates=['날짜'])
buzz  = pd.read_csv('data/온라인버즈_2024_2025.csv', parse_dates=['날짜'])

print('매체노출:', media.shape, '| 하루당 줄 수(예):', media.groupby('날짜').size().mode().tolist())
display(media.head(4))
print('온라인버즈:', buzz.shape, '| 하루당 줄 수(예):', buzz.groupby('날짜').size().mode().tolist())
display(buzz.head(3))

---
# Part 1 · 결합 — 그레인 맞춰 하나로

## ✏️ 연습 1 — 그레인 맞추기 : 매체노출을 '하루 단위'로 집계

매체노출은 하루에 채널별로 여러 줄입니다. 버즈(하루 1줄)와 붙이려면 먼저 **하루 단위로 집계**해야 해요.
`날짜`·`캠페인ID`별로 **총게시건수·총도달수**를 구하세요.
> 힌트: `media.groupby(['날짜','캠페인ID'], as_index=False).agg(총게시건수=('게시건수','sum'), 총도달수=('도달수','sum'))`

In [ ]:
# 💡 풀이
media_daily = media.groupby(['날짜','캠페인ID'], as_index=False).agg(
    총게시건수=('게시건수','sum'),
    총도달수=('도달수','sum'),
)
print('원본 매체노출:', media.shape, '→ 하루 단위 집계 후:', media_daily.shape)  # 2925줄 → 731줄
media_daily.head()

## ✏️ 연습 2 — `merge` : 일별 매체노출 + 버즈 결합

이제 하루 단위가 된 `media_daily`와 `buzz`를 **날짜·캠페인ID**로 결합하세요. 둘 다 하루 1줄이니 **1:1** 결합입니다.
> 힌트: `media_daily.merge(buzz, on=['날짜','캠페인ID'], how='inner', validate='one_to_one')`
> — `validate`로 정말 1:1인지 자동 점검!

In [ ]:
# 💡 풀이
mkt = media_daily.merge(buzz, on=['날짜','캠페인ID'], how='inner', validate='one_to_one')
print('통합 마케팅 일별표:', mkt.shape)   # 731행 (하루 = 1행)
mkt.head()

---
# Part 2 · 구조변환 — 요약표와 감정 펼치기

## ✏️ 연습 3 — `pivot_table` : 채널 × 월 도달수 요약표

매체노출 원본으로 **채널(행) × 월(열) 도달수 합계**표를 만드세요. 어느 채널이 큰지 한눈에 보입니다.
> 힌트: 먼저 월 컬럼 만들기 `media['월'] = media['날짜'].dt.to_period('M').astype(str)`
> 그다음 `pivot_table(index='채널구분', columns='월', values='도달수', aggfunc='sum', fill_value=0)`

In [ ]:
# 💡 풀이
media['월'] = media['날짜'].dt.to_period('M').astype(str)
ch_month = media.pivot_table(index='채널구분', columns='월', values='도달수', aggfunc='sum', fill_value=0)

print('▶ 채널 × 월 도달수')
display(ch_month)

# 히트맵으로 보기 (진할수록 도달수 큼) — SNS·온라인뉴스가 크고, 방송·지면/블로그는 작음
plt.figure(figsize=(12,3))
sns.heatmap(ch_month, cmap='YlOrRd')
plt.title('채널별 월 도달수'); plt.tight_layout(); plt.show()

## ✏️ 연습 4 — `melt` : 감정 비율 3종을 한 열로

버즈에는 `긍정_비율`·`부정_비율`·`중립_비율`이 **각각 다른 열**로 흩어져 있습니다. `melt`로 **감정·비율** 두 열의 Long으로 모으세요.
> 힌트: `buzz.melt(id_vars=['날짜'], value_vars=['긍정_비율','부정_비율','중립_비율'], var_name='감정', value_name='비율')

In [ ]:
# 💡 풀이
senti = buzz.melt(id_vars=['날짜'],
                  value_vars=['긍정_비율','부정_비율','중립_비율'],
                  var_name='감정', value_name='비율')
display(senti.head(6))

# Long으로 모았더니 감정별 평균을 한 줄로 구할 수 있음
print('감정별 평균 비율(%):')
print(senti.groupby('감정')['비율'].mean().round(1))

## ✏️ 연습 5 — 인사이트 : 노출이 성과로 이어질까?

결합해서 만든 `mkt`(일별 통합표)로, **도달수(노출)** 와 성과 지표(**유입수·버즈량·긍정비율**)의 **상관관계**를 확인하고 한 문장으로 해석하세요.
> 힌트: `mkt['총도달수'].corr(mkt['유입수'])` — 1에 가까울수록 강한 양의 관계

In [ ]:
# 💡 풀이
print('총도달수 ↔ 유입수  상관:', round(mkt['총도달수'].corr(mkt['유입수']), 3))
print('총도달수 ↔ 버즈량  상관:', round(mkt['총도달수'].corr(mkt['버즈량']), 3))
print('버즈량   ↔ 긍정비율 상관:', round(mkt['버즈량'].corr(mkt['긍정_비율']), 3))

# 도달수 vs 유입수 산점도
sns.scatterplot(data=mkt, x='총도달수', y='유입수', alpha=0.4)
plt.title('도달수 vs 유입수'); plt.tight_layout(); plt.show()

### ✍️ 인사이트 (예시 답)

> 예시: "**도달수가 늘면 유입수도 뚜렷이 는다(상관 약 0.82)** — 매체 노출이 사이트 유입으로 잘 이어진다. 버즈량과도 어느 정도 관계(약 0.56)가 있지만, **버즈량이 많다고 긍정비율이 높아지진 않았다(상관 ≈ 0)** — '많이 언급됨'과 '좋게 언급됨'은 다른 문제다."
>
> ※ 표·상관계수에서 읽히는 사실을 근거로 한 문장이면 정답입니다.

---
## ✅ 정리

| 단계 | 한 일 | 핵심 |
|---|---|---|
| 결합 ① | 매체노출을 **하루 단위로 집계** | 그레인 맞추기 (하루 4줄 → 1줄) |
| 결합 ② | 버즈와 **1:1 `merge`** | `validate='one_to_one'`로 점검 |
| 구조변환 ③ | 채널 × 월 **`pivot_table`** | 채널별 규모 비교 |
| 구조변환 ④ | 감정 3열 **`melt`** | 흩어진 값을 한 열로 → 집계 쉬움 |
| 인사이트 ⑤ | 도달수 ↔ 성과 **상관** | 노출 → 유입은 강함, 버즈 → 긍정은 무관 |

> 핵심: **그레인이 다른 데이터는 집계로 단위를 맞춘 뒤 결합한다.** 이게 실무 결합에서 가장 자주 만나는 함정입니다.